# Policy index setup (Search-managed Standard ingestion)

Search-managed Standard ingestion combines Content Understanding (CU) layout extraction and figure understanding, optimized for search. The current demo's directly populated CU Layout index uses layout-only extraction; it doesn't include the managed pipeline's model-generated figure descriptions.

Use the existing Azure AI Search service; this notebook creates no Search service, knowledge base or hosted agent. The agent queries an ordinary index directly.

Pipeline: policy PDFs -> Blob Storage -> Search indexer -> Content Understanding skill -> policy index -> Agent Framework.

Before starting, run `infra\setup.ps1` from the repository root and sign in with `az login`. Select `.tools\policy-venv\Scripts\python.exe` as the notebook's Python environment (your notebook host may prompt to install its kernel support). To use a separately managed kernel, install `infra/requirements-policy-agent-framework.txt` in that environment. Agent Framework is installed through pip, not npm; its core and OpenAI packages have independent version numbers. You need an existing Foundry model deployment and CU claim analyzer. The Search identity needs Storage Blob Data Reader on the policy storage account and Cognitive Services User on Foundry. Setup reports missing permissions but never assigns roles or changes networking.

The notebook uses the same functions as `infra/setup-policy-index.py`. Only the explicit apply cell can upload PDFs or create ingestion objects.

In [ ]:
import importlib.util, json
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'infra' / 'setup-policy-index.py').exists())
spec = importlib.util.spec_from_file_location('policy_index_setup', root / 'infra' / 'setup-policy-index.py')
setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(setup)
http = setup.Http()

# Provision .tools/policy-deployment.json as described in docs/policy-setup.md.
# The analyzer hash is read from the existing analyzer; never enter it manually.
SEED = {}  # Optional overrides; resource IDs and model/analyzer names come from deployment settings.
CREATE_CLAIM_ANALYZER = False  # Explicit opt-in for a fresh lab; never replaces an existing analyzer.

## 1. Check sources and prerequisites

Deployment settings are machine-local, not hardcoded in the scripts or notebook. The analyzer definition is read from CU, so no hash needs to be entered. Preflight validates PDF hashes, citation mappings, resource access and roles. It changes nothing in Azure.

In [ ]:
report = setup.preflight(http, root, seed=SEED, create_claim_analyzer=CREATE_CLAIM_ANALYZER)
print(json.dumps({k: report.get(k) for k in ('ready', 'documents', 'names', 'claimAnalyzer', 'missingRoles', 'problems', 'existing')}, indent=2))

## 2. Inspect the managed pipeline

The CU skill uses the Search preview API for semantic chunks, table Markdown, page ranges and model-generated figure descriptions. `contentExtractionMode: standard` is a knowledge-source setting, not an index property; this pipeline configures the CU skill directly instead.

Only registered policy PDFs are ingested. Answer guides, claim packets and editable duplicates aren't policy knowledge. The generation name changes when sources or extraction settings change.

In [ ]:
sources = setup.load_sources(root)
settings = setup.runtime_settings(root, SEED)
generation = setup.generation_hash(sources, settings)
names = setup.names_for(generation)
print(json.dumps({'sourceFiles': [d['file'] for d in sources['documents']],
                  'names': names, 'skillset': setup.skillset_definition(names, generation, settings)}, indent=2))

## 3. Apply explicitly and activate only after verification

Set `APPLY = True` only after resolving the preflight report. This can incur Blob, Search, CU and model charges. Apply checks prerequisites again, uploads the source PDFs, creates generation-scoped objects, waits for ingestion and verifies every source plus state-filtered retrieval. Only then is the workbench configuration switched to the new index. The existing working configuration remains active on failure.

Changed sources automatically create a new generation. `REFRESH = True` resets and reprocesses an existing generation; it isn't necessary for an ordinary source update. No previous Azure resources are deleted.

In [ ]:
APPLY = False
REFRESH = False
if APPLY:
    print(json.dumps(setup.apply(http, root, refresh=REFRESH, seed=SEED, create_claim_analyzer=CREATE_CLAIM_ANALYZER), indent=2))
else:
    print('APPLY is False: preflight only')

## 4. Compare the agent results

After successful activation, load saved Parse and extract results in the workbench and select **Check policies** > **FoundryIQ**. FoundryIQ is the lab's umbrella label; this implementation queries Azure AI Search directly, not a Foundry IQ knowledge base. Select **Run live** to request policy work; matching current reviews are reused. An uncached packet interprets rental applicability once, then runs three parallel claim-versus-policy comparisons: **Daily rate**, **Rental start**, and **Effective terms**. These calls reuse saved typed fields and relevant native page text without another claim CU extraction or a mandatory claim-summary cache. A satisfied comparison is not payment approval. **No policy retrieval** only displays a saved baseline if available. Inspect linked claim/policy values in the right panel, and use **Ask agent** for follow-ups.

Use the matched policy's eligibility figure, sparse table cells and scanned records to inspect extraction quality. TX/WA figures control only for their own policy and insured VIN; other figures are illustrative and written clauses control. Figure descriptions are model-generated, not a guarantee of correct interpretation: ambiguous policy evidence still requires review. Missing information genuinely needed for the selected comparison shows **Waiting for info**, not an agent still running. The Original, Prebuilt Search and Prebuilt Digital Parse tabs are separate document representations; the saved review retains its actual index provenance.